# Mouse brain MERFISH figure revisions

This notebook redraws selected Microglia mouse brain MERFISH figures with larger typography and publication-style colors. The source analysis directory is treated as read-only; revised PNG/PDF pairs are written to `FigureReproducing/mouse brain`.

Purpose: support the main-text mouse brain result that stGP identifies aging-associated microglial programs and spatial/proximity structure, especially the late-life Microglia stGP2 evidence.

In [1]:
from pathlib import Path
import json
import os
import sys

import anndata as ad
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial import cKDTree
from scipy.stats import mannwhitneyu
import scipy.sparse as sp
from sklearn.cluster import SpectralClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

BASE = Path(os.environ.get('STGP_REPRO_ROOT', Path.cwd().resolve()))
if not (BASE / 'FigureReproducing').exists():
    BASE = BASE.parent
FIG_REPRO_DIR = BASE / 'FigureReproducing'
MOUSE_DIR = BASE / 'RealData_MouseBrainMERFISH'
OUT_DIR = FIG_REPRO_DIR / 'mouse brain'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for pth in (MOUSE_DIR, FIG_REPRO_DIR):
    pth_str = str(pth)
    if pth_str not in sys.path:
        sys.path.insert(0, pth_str)
from plots import (
    METHOD_COLORS,
    PANEL_CMAPS,
    VarPartColors,
    _bg_per_mouse,
    _select_mice_by_target_ages,
    load_method,
    plot_enrichment_panel,
    plot_spacetime_embedding_stack,
)
from plot import (
    DPI,
    draw_alpha_ci,
    ordered_stgp_alpha,
    p_to_stars,
    save_pair as save_figure_pair,
    setup_publication_style,
)

TARGET_AGES = (6.6, 18.8, 24.6, 34.5)
METHOD_ORDER = ['stGP', 'SpatialPCA', 'MEFISTO', 'STAMP', 'Popari']
CLUSTER_LABEL_COLORS = {
    1: '#1f78b4',  # blue
    2: '#e31a1c',  # red
    3: '#9edae5',  # light cyan
    4: '#f4a6c8',  # light pink
    5: '#33a02c',
    6: '#ff7f00',
    7: '#6a3d9a',
    8: '#b15928',
}


def cluster_color(label):
    try:
        key = int(label)
    except (TypeError, ValueError):
        key = str(label)
    return CLUSTER_LABEL_COLORS.get(key, '#7f7f7f')

setup_publication_style('mouse_brain')

print(f'Revised figures will be saved to: {OUT_DIR}')

Revised figures will be saved to: /home/byual/stGP-0529/FigureReproducing/mouse brain


In [ ]:
def save_pair(fig, stem, *, out_dir=OUT_DIR, bbox_inches='tight', pad_inches=0.05):
    return save_figure_pair(
        fig,
        stem,
        out_dir=out_dir,
        dpi=DPI,
        bbox_inches=bbox_inches,
        pad_inches=pad_inches,
    )


MOUSE_FULL_QC_PATH = MOUSE_DIR / 'data' / 'qc' / 'aging_coronal_qc.h5ad'


def _load_baselines_enabled():
    return os.environ.get('STGP_LOAD_BASELINES', '1').lower() not in {'0', 'false', 'no'}


def _load_optional_h5ad(path, *, label):
    try:
        return ad.read_h5ad(path)
    except (FileNotFoundError, OSError) as exc:
        print(f'[skip] {label} is unavailable: {path} ({exc})')
        return None


def load_optional_mouse_full_adata():
    return _load_optional_h5ad(MOUSE_FULL_QC_PATH, label='Full mouse QC AnnData')


def _load_optional_method(method, result_dir, *, celltype='Microglia'):
    try:
        return load_method(method, result_dir, celltype=celltype)
    except (FileNotFoundError, OSError) as exc:
        print(f'[skip] {method} baseline result is unavailable: {result_dir} ({exc})')
        return None


def load_microglia_results(include_baselines=None):
    if include_baselines is None:
        include_baselines = _load_baselines_enabled()

    results = {
        'stGP': load_method('stGP', MOUSE_DIR / 'Results' / 'stgp' / 'Microglia', celltype='Microglia'),
    }
    if not include_baselines:
        print('[skip] Baseline result loading disabled by STGP_LOAD_BASELINES=0.')
        return results

    baseline_specs = {
        'MEFISTO': MOUSE_DIR / 'Results' / 'baselines' / 'mefisto' / 'Microglia',
        'Popari': MOUSE_DIR / 'Results' / 'baselines' / 'popari' / 'Microglia',
        'SpatialPCA': MOUSE_DIR / 'Results' / 'baselines' / 'spatialpca' / 'Microglia',
        'STAMP': MOUSE_DIR / 'Results' / 'baselines' / 'stamp' / 'Microglia',
    }
    for method, result_dir in baseline_specs.items():
        loaded = _load_optional_method(method, result_dir)
        if loaded is not None:
            results[method] = loaded
    return results


def plot_alpha_over_age_large(stgp):
    info = stgp.adata.uns.get('stgp', {})
    prog_names = stgp.scores.columns.astype(str).tolist()
    idx = prog_names.index('stGP2') if 'stGP2' in prog_names else 1
    ages, y, lo, hi, _ = ordered_stgp_alpha(info, idx)

    fig, ax = plt.subplots(figsize=(6.5, 5.05), constrained_layout=True)
    draw_alpha_ci(ax, ages, y, lo, hi, color='#2C7FB8', scatter_s=72)
    ax.set_xlabel('Age (months)', fontsize=24)
    ax.set_ylabel(r'Aging effect $\alpha$', fontsize=24)
    ax.tick_params(axis='both', labelsize=20, length=4.5, width=1.1)
    if lo is not None and hi is not None:
        ax.legend(fontsize=20, loc='best')
    save_pair(fig, 'Microglia_stGP_stGP2_alpha_over_age')



def load_stgp_varpart_from_results():
    import pickle

    result_dir = MOUSE_DIR / 'Results' / 'stgp' / 'Microglia'
    with open(result_dir / 'stgp_result.pkl', 'rb') as f:
        stgp_result = pickle.load(f)

    theta = np.asarray(stgp_result['theta'], dtype=float)
    labels = pd.read_csv(result_dir / 'W.csv', index_col=0).index.astype(str).tolist()
    if len(labels) != theta.shape[0]:
        labels = [f'stGP{i + 1}' for i in range(theta.shape[0])]

    total = theta[:, 0] + theta[:, 1]
    total = np.where(total > 0, total, 1.0)
    return pd.DataFrame(
        {
            'sigma2_age': theta[:, 0] / total,
            'tau2_spa': theta[:, 1] / total,
        },
        index=labels,
    )



def plot_variance_partition_large():
    df = load_stgp_varpart_from_results()
    colors = VarPartColors()
    if 'sigma2_age' in df.columns:
        components = [c for c in ['sigma2_age', 'tau2_spa', 'sigma2_e'] if c in df.columns]
        labels = {'sigma2_age': r'$\sigma^2_\mathrm{age}$', 'tau2_spa': r'$\tau^2_\mathrm{spa}$', 'sigma2_e': r'$\sigma^2_e$'}
        comp_colors = {'sigma2_age': colors.age, 'tau2_spa': colors.region, 'sigma2_e': colors.residuals}
    else:
        components = [c for c in ['Age', 'Region', 'Both', 'Residuals'] if c in df.columns]
        labels = {c: c for c in components}
        comp_colors = {'Age': colors.age, 'Region': colors.region, 'Both': colors.both, 'Residuals': colors.residuals}

    programs = df.index.astype(str).tolist()
    y = np.arange(len(programs))
    fig, ax = plt.subplots(figsize=(8.0, max(4.2, 0.92 * len(programs) + 2.0)), constrained_layout=True)
    left = np.zeros(len(programs))
    for comp in components:
        vals = df[comp].fillna(0).clip(lower=0).to_numpy(float) * 100.0
        ax.barh(y, vals, left=left, height=0.58, color=comp_colors[comp], label=labels[comp], linewidth=0)
        for i, (v, lft) in enumerate(zip(vals, left)):
            if v > 6:
                ax.text(lft + v / 2, y[i], f'{v:.0f}%', ha='center', va='center', fontsize=14, color='white', fontweight='bold')
        left += vals
    ax.set_yticks(y)
    ax.set_yticklabels(programs, fontsize=24)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xticklabels(['0', '25', '50', '75', '100%'], fontsize=20)
    ax.set_xlabel('')
    ax.set_title('Variance explained (%)', fontsize=34, fontweight='normal', pad=16)
    ax.spines['left'].set_visible(False)
    ax.tick_params(left=False, axis='x', length=4.5, width=1.0)
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.28), ncol=len(components), fontsize=22, handlelength=1.2, columnspacing=1.0)
    save_pair(fig, 'Microglia_stGP_variance_partition')


def _values_for_spec(method_result, col_idx, *, use_spatial_obsm):
    if use_spatial_obsm and 'X_stgp_spatial' in method_result.adata.obsm:
        arr = np.asarray(method_result.adata.obsm['X_stgp_spatial'])
        return arr[:, col_idx]
    return method_result.scores.iloc[:, col_idx].to_numpy(dtype=float)


def _limits(vals, *, signed):
    vals = np.asarray(vals, dtype=float)
    finite = vals[np.isfinite(vals)]
    if finite.size == 0:
        return 0.0, 1.0
    if signed:
        vmax = float(np.nanpercentile(np.abs(finite), 99))
        vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1.0
        return -vmax, vmax
    vmin = float(np.nanpercentile(finite, 1))
    vmax = float(np.nanpercentile(finite, 99))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or np.isclose(vmin, vmax):
        return float(np.nanmin(finite)), float(np.nanmax(finite) + 1e-9)
    return vmin, vmax


def draw_spatial_group(fig, subplotspec, method, values, title, *, results, adata_full, signed, title_fontsize=32, age_fontsize=24, cbar_fontsize=15, cbar_ticksize=13, fg_dot_size=7.0, bg_dot_size=0.35):
    m = results[method]
    adata = m.adata
    obs = adata.obs
    sp_xy = np.asarray(adata.obsm['spatial'])
    mouse_ids = obs['mouse_id'].astype(str).to_numpy()
    selected = _select_mice_by_target_ages(obs, TARGET_AGES)
    bg_by_mouse = _bg_per_mouse(adata_full, mouse_ids)
    vmin, vmax = _limits(values, signed=signed)
    cmap = 'RdBu_r' if signed else 'YlOrBr'

    gs = subplotspec.subgridspec(3, 3, height_ratios=[0.20, 1, 1], width_ratios=[1, 1, 0.075], wspace=0.035, hspace=0.17)
    title_ax = fig.add_subplot(gs[0, :2])
    title_ax.axis('off')
    title_ax.text(0.5, 0.50, title, ha='center', va='center', fontsize=title_fontsize, fontweight='normal')
    axes = [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1])]
    cbar_ax = fig.add_subplot(gs[1:, 2])

    sc_ref = None
    for ax, (mid, age, _target_age) in zip(axes, selected):
        if mid in bg_by_mouse:
            bg = bg_by_mouse[mid]
            ax.scatter(bg[:, 0], bg[:, 1], c='#D8D8D8', s=bg_dot_size, linewidths=0, rasterized=True, zorder=1)
        mask = mouse_ids == mid
        sc_ref = ax.scatter(
            sp_xy[mask, 0], sp_xy[mask, 1],
            c=np.asarray(values)[mask], cmap=cmap, vmin=vmin, vmax=vmax,
            s=fg_dot_size, linewidths=0, rasterized=True, zorder=2,
        )
        ax.set_aspect('equal')
        ax.set_title(f'{age:.1f} mo', fontsize=age_fontsize, pad=3)
        ax.axis('off')
    sm = sc_ref if sc_ref is not None else ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap=cmap)
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label('score', fontsize=cbar_fontsize, labelpad=8)
    cbar.ax.tick_params(labelsize=cbar_ticksize, length=3, width=0.9, pad=2)
    return axes


def plot_selected_spatial_panels(results, adata_full):
    specs = [
        ('MEFISTO', 2, 'MEFISTO_prog3_spatial_2x2', 'MEFISTO', False, True),
        ('stGP', 1, 'Microglia_stGP_stGP2_spatial_2x2', 'stGP', True, True),
        ('Popari', 3, 'Popari_prog4_spatial_2x2', 'Popari', False, False),
        ('SpatialPCA', 2, 'SpatialPCA_prog3_spatial_2x2', 'SpatialPCA', False, True),
        ('STAMP', 2, 'STAMP_prog3_spatial_2x2', 'STAMP', False, False),
    ]
    plotted = {}
    for method, col_idx, stem, title, use_spatial, signed in specs:
        if method not in results:
            print(f'[skip] {stem}: {method} result is not loaded.')
            continue
        if col_idx >= results[method].scores.shape[1]:
            print(f'[skip] {stem}: {method} has only {results[method].scores.shape[1]} programs.')
            continue
        vals = _values_for_spec(results[method], col_idx, use_spatial_obsm=use_spatial)
        fig = plt.figure(figsize=(7.2, 6.35), constrained_layout=False)
        outer = fig.add_gridspec(1, 1, left=0.02, right=0.965, top=0.965, bottom=0.035)
        draw_spatial_group(fig, outer[0], method, vals, title, results=results, adata_full=adata_full, signed=signed)
        plt.close(fig)
        plotted[method] = (vals, signed)
    return plotted


def plot_baseline_spatial_composite(results, adata_full, plotted):
    order = [('SpatialPCA', 'SpatialPCA'), ('Popari', 'Popari'), ('STAMP', 'STAMP'), ('MEFISTO', 'MEFISTO')]
    available = [(method, title) for method, title in order if method in plotted]
    if not available:
        print('[skip] Microglia baseline composite: no baseline spatial panels were available.')
        return

    ncols = min(2, len(available))
    nrows = int(np.ceil(len(available) / ncols))
    fig = plt.figure(figsize=(6.7 * ncols, 5.6 * nrows), constrained_layout=False)
    outer = fig.add_gridspec(nrows, ncols, left=0.012, right=0.988, top=0.988, bottom=0.012, wspace=0.065, hspace=0.155)
    for i, (method, title) in enumerate(available):
        spec = outer[i // ncols, i % ncols] if nrows > 1 or ncols > 1 else outer[0, 0]
        vals, signed = plotted[method]
        draw_spatial_group(
            fig, spec, method, vals, title, results=results, adata_full=adata_full, signed=signed,
            title_fontsize=25, age_fontsize=17, cbar_fontsize=13, cbar_ticksize=11,
            fg_dot_size=5.8, bg_dot_size=0.25,
        )
    save_pair(fig, 'Microglia_baseline_methods_spatial_2x2_composite', bbox_inches='tight', pad_inches=0.025)


def pval_to_stars(pval):
    return p_to_stars(pval, nan_label='n.s.', nonsig_label='n.s.')


def plot_near_far_violin_tcell():
    adata_mg = ad.read_h5ad(MOUSE_DIR / 'Results' / 'stgp' / 'Microglia' / 'adata_with_scores.h5ad')
    adata_all = load_optional_mouse_full_adata()
    if adata_all is None:
        print('[skip] stGP2_near_far_violin_Tcell: requires full mouse QC AnnData for T cell proximity.')
        return
    target_age = np.asarray(adata_mg.obs['age'], dtype=float)
    target_coord = np.asarray(adata_mg.obsm['spatial'])
    target_b = np.asarray(adata_mg.obsm['X_stgp_spatial'])
    glob_age = np.asarray(adata_all.obs['age'], dtype=float)
    glob_ct = np.asarray(adata_all.obs['celltype']).astype(str)
    glob_coord = np.asarray(adata_all.obsm['spatial'])
    ages = np.unique(target_age)
    r_near, r_far = 50.0, 150.0
    program_k = 1
    effector = 'T cell'
    near_color = METHOD_COLORS['stGP']
    far_color = '#7F7F7F'
    eff_xy_per_age = {float(age): glob_coord[(glob_age == age) & (glob_ct == effector)] for age in ages}

    def compute_near_far(target_mask, effector_xy_by_age):
        all_near, all_far = [], []
        for age in np.unique(target_age[target_mask]):
            eff_xy = effector_xy_by_age.get(float(age))
            if eff_xy is None or len(eff_xy) < 1:
                continue
            blk = target_mask & (target_age == age)
            tg_xy = target_coord[blk]
            if len(tg_xy) < 1:
                continue
            d, _ = cKDTree(eff_xy).query(tg_xy, k=1)
            near = d <= r_near
            far = d > r_far
            if near.sum() < 5 or far.sum() < 5:
                continue
            b_vals = target_b[blk, program_k]
            all_near.append(b_vals[near])
            all_far.append(b_vals[far])
        if not all_near:
            return None
        return np.concatenate(all_near), np.concatenate(all_far)

    def render_violin_pair(ax, v_near, v_far, title):
        parts = ax.violinplot([v_near, v_far], positions=[0, 1], showmedians=False, showextrema=False, widths=0.74)
        for pc, col in zip(parts['bodies'], [near_color, far_color]):
            pc.set_facecolor(col)
            pc.set_edgecolor(col)
            pc.set_alpha(0.86)
            pc.set_linewidth(1.0)
        ax.boxplot([v_near, v_far], positions=[0, 1], widths=0.16, patch_artist=True,
                   boxprops=dict(facecolor='white', edgecolor='black', linewidth=1.1),
                   medianprops=dict(color='black', linewidth=1.6),
                   whiskerprops=dict(color='black', linewidth=1.1),
                   capprops=dict(color='black', linewidth=1.1), showfliers=False)
        try:
            _, pval = mannwhitneyu(v_near, v_far, alternative='two-sided')
        except ValueError:
            pval = np.nan
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Near', 'Far'], fontsize=20, fontweight='normal')
        ax.set_title(f'{title} ({pval_to_stars(pval)})', fontsize=21, fontweight='normal', pad=10)
        ax.axhline(0, ls='--', color='black', alpha=0.42, linewidth=0.95)
        ax.set_ylabel('stGP2 spatial embedding', fontsize=18)
        ax.tick_params(axis='y', labelsize=16, length=4, width=1.0)
        ax.tick_params(axis='x', length=0, pad=6)

    median_age = float(np.median(ages))
    young_ages = sorted(float(a) for a in ages if a < median_age)
    old_ages = sorted(float(a) for a in ages if a >= median_age)
    fig, axes = plt.subplots(2, 1, figsize=(5.2, 8.5), sharey=False, constrained_layout=False)
    fig.subplots_adjust(left=0.23, right=0.97, top=0.96, bottom=0.08, hspace=0.32)
    for ax, target_mask, group_ages, label in zip(
        axes,
        [np.isin(target_age, young_ages), np.isin(target_age, old_ages)],
        [young_ages, old_ages],
        ['Young slices', 'Old slices'],
    ):
        res = compute_near_far(target_mask, {a: eff_xy_per_age[a] for a in group_ages})
        if res is None:
            ax.text(0.5, 0.5, 'Insufficient cells', transform=ax.transAxes, ha='center', va='center', fontsize=16, color='#777777')
            ax.set_axis_off()
            continue
        render_violin_pair(ax, res[0], res[1], label)
    save_pair(fig, 'stGP2_near_far_violin_Tcell')


def _embedding_for_method(m):
    if m.method == 'stGP' and 'X_stgp_spatial' in m.adata.obsm:
        emb = np.asarray(m.adata.obsm['X_stgp_spatial'], dtype=float)
    elif m.method == 'SpatialPCA' and 'X_spatialpca' in m.adata.obsm:
        emb = np.asarray(m.adata.obsm['X_spatialpca'], dtype=float)
    else:
        emb = m.scores.to_numpy(dtype=float)
    return np.nan_to_num(emb, nan=0.0, posinf=0.0, neginf=0.0)


def _spectral_knn_labels(X, n_clusters, *, random_state=42):
    X = np.asarray(X, dtype=float)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    n = X.shape[0]
    if n <= n_clusters:
        return np.arange(n, dtype=int) % max(n_clusters, 1)
    Xs = StandardScaler().fit_transform(X)
    k_nn = min(max(2, int(np.round(np.sqrt(n)))), n - 1)
    nn = NearestNeighbors(n_neighbors=k_nn + 1, metric='euclidean').fit(Xs).kneighbors(return_distance=False)[:, 1:]
    rows = np.repeat(np.arange(nn.shape[0]), k_nn)
    cols = nn.ravel()
    graph = sp.csr_matrix((np.ones(rows.size), (rows, cols)), shape=(n, n))
    graph = graph.maximum(graph.T)
    return SpectralClustering(n_clusters=n_clusters, affinity='precomputed', assign_labels='kmeans', random_state=random_state).fit_predict(graph)


def _cluster_labels_for_mouse(m, mouse_id):
    emb = _embedding_for_method(m)
    mouse_ids = m.adata.obs['mouse_id'].astype(str).to_numpy()
    mask = mouse_ids == str(mouse_id)
    if m.method == 'STAMP':
        labels = np.argmax(emb, axis=1).astype(int) + 1
        return labels[mask]
    region_labels = m.adata.obs['region'].astype(str).to_numpy() if 'region' in m.adata.obs.columns else None
    n_clusters = max(2, len(np.unique(region_labels)) if region_labels is not None else 6)
    n_eff = min(max(2, n_clusters), max(1, int(mask.sum()) - 1))
    return _spectral_knn_labels(emb[mask], n_eff, random_state=42) + 1


def plot_benchmark_clustering_mouse101(results, adata_full):
    mouse_id = '101'
    age_label = 34.5
    method_order = ['SpatialPCA', 'MEFISTO', 'STAMP', 'Popari']
    available = [method for method in method_order if method in results]
    if not available:
        print('[skip] Microglia benchmark clustering: no baseline results were loaded.')
        return

    ncols = min(2, len(available))
    nrows = int(np.ceil(len(available) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.95 * ncols, 3.78 * nrows), constrained_layout=False, squeeze=False)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.94, bottom=0.02, wspace=0.025, hspace=0.10)
    flat_axes = axes.ravel()
    for ax in flat_axes[len(available):]:
        ax.axis('off')
    for ax, method in zip(flat_axes, available):
        m = results[method]
        labels = _cluster_labels_for_mouse(m, mouse_id)
        obs = m.adata.obs
        xy = np.asarray(m.adata.obsm['spatial'])
        mouse_ids = obs['mouse_id'].astype(str).to_numpy()
        mask = mouse_ids == mouse_id
        bg_by_mouse = _bg_per_mouse(adata_full, mouse_ids)
        if mouse_id in bg_by_mouse:
            bg = bg_by_mouse[mouse_id]
            ax.scatter(bg[:, 0], bg[:, 1], c='#E0E0E0', s=0.36, linewidths=0, rasterized=True, zorder=1)
        colors = [cluster_color(lbl) for lbl in labels]
        ax.scatter(xy[mask, 0], xy[mask, 1], c=colors, s=8.5, linewidths=0, rasterized=True, zorder=2)
        ax.set_aspect('equal')
        ax.set_title(method, fontsize=24, fontweight='normal', pad=4)
        ax.axis('off')
    save_pair(fig, f'Microglia_benchmark_clustering_mouse_{mouse_id}_age_{age_label:.1f}mo', bbox_inches=None, pad_inches=0)

In [3]:
def plot_near_far_violin_tcell_horizontal():
    adata_mg = ad.read_h5ad(MOUSE_DIR / 'Results' / 'stgp' / 'Microglia' / 'adata_with_scores.h5ad')
    adata_all = load_optional_mouse_full_adata()
    if adata_all is None:
        print('[skip] stGP2_near_far_violin_Tcell_horizontal: requires full mouse QC AnnData for T cell proximity.')
        return
    target_age = np.asarray(adata_mg.obs['age'], dtype=float)
    target_coord = np.asarray(adata_mg.obsm['spatial'])
    target_b = np.asarray(adata_mg.obsm['X_stgp_spatial'])
    glob_age = np.asarray(adata_all.obs['age'], dtype=float)
    glob_ct = np.asarray(adata_all.obs['celltype']).astype(str)
    glob_coord = np.asarray(adata_all.obsm['spatial'])
    ages = np.unique(target_age)
    r_near, r_far = 50.0, 150.0
    program_k = 1
    effector = 'T cell'
    near_color = METHOD_COLORS['stGP']
    far_color = '#7F7F7F'
    eff_xy_per_age = {float(age): glob_coord[(glob_age == age) & (glob_ct == effector)] for age in ages}

    def compute_near_far(target_mask, effector_xy_by_age):
        all_near, all_far = [], []
        for age in np.unique(target_age[target_mask]):
            eff_xy = effector_xy_by_age.get(float(age))
            if eff_xy is None or len(eff_xy) < 1:
                continue
            blk = target_mask & (target_age == age)
            tg_xy = target_coord[blk]
            if len(tg_xy) < 1:
                continue
            d, _ = cKDTree(eff_xy).query(tg_xy, k=1)
            near = d <= r_near
            far = d > r_far
            if near.sum() < 5 or far.sum() < 5:
                continue
            b_vals = target_b[blk, program_k]
            all_near.append(b_vals[near])
            all_far.append(b_vals[far])
        if not all_near:
            return None
        return np.concatenate(all_near), np.concatenate(all_far)

    def render_violin_pair(ax, v_near, v_far, title):
        parts = ax.violinplot([v_near, v_far], positions=[0, 1], showmedians=False, showextrema=False, widths=0.74)
        for pc, col in zip(parts['bodies'], [near_color, far_color]):
            pc.set_facecolor(col)
            pc.set_edgecolor(col)
            pc.set_alpha(0.86)
            pc.set_linewidth(1.0)
        ax.boxplot([v_near, v_far], positions=[0, 1], widths=0.16, patch_artist=True,
                   boxprops=dict(facecolor='white', edgecolor='black', linewidth=1.1),
                   medianprops=dict(color='black', linewidth=1.6),
                   whiskerprops=dict(color='black', linewidth=1.1),
                   capprops=dict(color='black', linewidth=1.1), showfliers=False)
        try:
            _, pval = mannwhitneyu(v_near, v_far, alternative='two-sided')
        except ValueError:
            pval = np.nan
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Near', 'Far'], fontsize=20, fontweight='normal')
        ax.set_title(f'{title} ({pval_to_stars(pval)})', fontsize=21, fontweight='normal', pad=10)
        ax.axhline(0, ls='--', color='black', alpha=0.42, linewidth=0.95)
        ax.tick_params(axis='y', labelsize=16, length=4, width=1.0)
        ax.tick_params(axis='x', length=0, pad=6)

    median_age = float(np.median(ages))
    young_ages = sorted(float(a) for a in ages if a < median_age)
    old_ages = sorted(float(a) for a in ages if a >= median_age)
    groups = [
        (np.isin(target_age, young_ages), young_ages, 'Young slices'),
        (np.isin(target_age, old_ages), old_ages, 'Old slices'),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.65), sharey=True, constrained_layout=False)
    fig.subplots_adjust(left=0.115, right=0.985, top=0.82, bottom=0.18, wspace=0.16)
    for ax, (target_mask, group_ages, label) in zip(axes, groups):
        res = compute_near_far(target_mask, {a: eff_xy_per_age[a] for a in group_ages})
        if res is None:
            ax.text(0.5, 0.5, 'Insufficient cells', transform=ax.transAxes, ha='center', va='center', fontsize=16, color='#777777')
            ax.set_axis_off()
            continue
        render_violin_pair(ax, res[0], res[1], label)
    axes[0].set_ylabel('stGP2 spatial embedding', fontsize=18)
    axes[1].set_ylabel('')
    save_pair(fig, 'stGP2_near_far_violin_Tcell_horizontal')

In [4]:
def plot_mouse97_slingshot_spatial_for_figure_drawing():
    h5ad_path = MOUSE_DIR / 'Results' / 'stgp' / 'Microglia' / 'downstream_cluster' / 'mouse_97_stgp_cluster_slingshot.h5ad'
    adata = ad.read_h5ad(h5ad_path)

    xy = np.asarray(adata.obsm['spatial'], dtype=float)
    pt = pd.to_numeric(adata.obs['slingPseudotime_1'], errors='coerce').to_numpy(float)
    labels = pd.Categorical(adata.obs['clusterlabel'].astype(str))
    cats = labels.categories.astype(str).tolist()

    x = xy[:, 0]
    y = -xy[:, 1]
    label_arr = labels.astype(str)

    fig = plt.figure(figsize=(7.9, 7.55), constrained_layout=False)
    gs = fig.add_gridspec(
        2, 2,
        left=0.01, right=0.99, top=0.90, bottom=0.02,
        width_ratios=[1.00, 0.50],
        height_ratios=[1, 1],
        wspace=0.025, hspace=0.30,
    )

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0], sharex=ax0, sharey=ax0)
    leg_ax = fig.add_subplot(gs[0, 1])
    cbar_gs = gs[1, 1].subgridspec(1, 3, width_ratios=[0.20, 0.07, 0.73], wspace=0.0)
    cax = fig.add_subplot(cbar_gs[0, 0])

    for label in cats:
        mask = label_arr == str(label)
        ax0.scatter(
            x[mask], y[mask],
            s=8.5, linewidths=0, rasterized=True,
            color=cluster_color(label),
            label=str(label),
        )

    scat = ax1.scatter(
        x, y, c=pt,
        s=8.5, cmap='viridis', linewidths=0, rasterized=True,
    )

    for ax in [ax0, ax1]:
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_xlim(x.min(), x.max())
        ax.set_ylim(y.max(), y.min())
    ax0.set_title('stGP', fontsize=32, fontweight='normal', pad=18)

    leg_ax.axis('off')
    legend_handles = [
        Line2D(
            [0], [0],
            marker='o', linestyle='none',
            markerfacecolor=cluster_color(label),
            markeredgecolor='none',
            markersize=18,
            label=str(label),
        )
        for label in cats
    ]
    leg_ax.legend(
        handles=legend_handles,
        title='cluster label',
        fontsize=24,
        title_fontsize=24,
        loc='upper left',
        bbox_to_anchor=(0.0, 0.98),
        borderaxespad=0,
        handletextpad=0.8,
        labelspacing=0.75,
    )

    cbar = plt.colorbar(scat, cax=cax)
    cbar.set_label('Slingshot pseudotime', fontsize=17, labelpad=10)
    cbar.ax.tick_params(labelsize=15, length=4, width=1.0)
    save_pair(
        fig,
        'mouse_97_slingshot_spatial',
        bbox_inches=None,
        pad_inches=0,
    )



def plot_microglia_stgp2_enrichment_for_figure_drawing():
    enrich_path = MOUSE_DIR / 'Results' / 'enrichment' / 'Microglia' / 'Microglia_stGP2_enrichment.csv'
    df = pd.read_csv(enrich_path)
    required = {'gene_set', 'Term', 'Combined Score', 'Adjusted P-value'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Enrichment CSV is missing required columns: {sorted(missing)}')

    gene_set_order = [
        'GO Biological process',
        'GO Molecular Function',
        'GO Cellular Component',
        'Cell-type signatures',
    ]
    gene_sets = [name for name in gene_set_order if name in set(df['gene_set'].astype(str))]
    gene_sets += [name for name in df['gene_set'].astype(str).drop_duplicates().tolist() if name not in gene_sets]

    fig, axes = plt.subplots(
        len(gene_sets), 1,
        figsize=(5.4, 2.5 * len(gene_sets)),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).flatten()

    for ax, set_name in zip(axes, gene_sets):
        subset = df[df['gene_set'].astype(str) == set_name].copy()
        cmap = PANEL_CMAPS.get(set_name, plt.get_cmap('Greys'))
        plot_enrichment_panel(
            subset,
            ax,
            set_name,
            cmap=cmap,
            n_top=6,
            padj_threshold=0.1,
        )

    save_pair(
        fig,
        'Microglia_stGP2_enrichment',
        bbox_inches='tight',
        pad_inches=0.05,
    )



def plot_microglia_stgp2_proximity_regression():
    coef_path = MOUSE_DIR / 'Results' / 'proximity' / 'Microglia' / 'stGP2' / 'variance_decomposition_coefs_full.csv'
    meta_path = MOUSE_DIR / 'Results' / 'proximity' / 'Microglia' / 'stGP2' / 'variance_decomposition_meta.json'
    df = pd.read_csv(coef_path)
    with open(meta_path) as f:
        meta = json.load(f)

    df = df[df['name'] != 'intercept'].copy().sort_values('coef', ascending=True)
    y_pos = np.arange(len(df))
    colors = ['#c0392b' if v > 0 else '#2980b9' for v in df['coef']]

    n_pred = max(int(meta.get('n_predictors', len(df))), 1)
    fig, ax = plt.subplots(figsize=(8.5, max(0.36 * n_pred + 1.4, 5.5)))
    ax.barh(
        y_pos,
        df['coef'].to_numpy(float),
        xerr=1.96 * df['se'].to_numpy(float),
        color=colors,
        edgecolor='black',
        lw=0.5,
        alpha=0.9,
        capsize=2,
    )
    ax.axvline(0, ls=':', color='black', alpha=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df['name'], fontsize=13)
    ax.set_xlabel('Coefficient', fontsize=14)
    ax.tick_params(axis='x', labelsize=12)
    ax.grid(True, axis='x', alpha=0.3)
    fig.tight_layout()
    save_pair(fig, 'Microglia_stGP2_proximity_regression', bbox_inches='tight', pad_inches=0.05)



def plot_microglia_stgp2_spacetime_stack(results=None, adata_full=None):
    stgp = results['stGP'] if results is not None else load_method('stGP', MOUSE_DIR / 'Results' / 'stgp' / 'Microglia', celltype='Microglia')
    if adata_full is None:
        adata_full = load_optional_mouse_full_adata()

    emb = np.asarray(stgp.adata.obsm['X_stgp_spatial'], dtype=float)
    if emb.shape[0] != stgp.adata.n_obs:
        emb = emb.T
    prog_names = stgp.scores.columns.astype(str).tolist()
    prog_idx = prog_names.index('stGP2') if 'stGP2' in prog_names else 1

    fig = plot_spacetime_embedding_stack(
        adata=stgp.adata,
        values=emb[:, prog_idx],
        adata_full=adata_full,
        value_label='stGP2 spatial embedding',
        color_scale='symmetric',
        elev=30,
    )
    save_pair(fig, 'Microglia_stGP_stGP2_spatial_embedding_stack_tilt30', bbox_inches='tight', pad_inches=0.05)

In [ ]:
results = load_microglia_results()
adata_full = load_optional_mouse_full_adata()

# 1. Larger stGP2 alpha-over-age panel; y-axis label changed to Aging effect.
plot_alpha_over_age_large(results['stGP'])

# 3. Larger variance partition panel with Variance explained (%) as the title.
plot_variance_partition_large()

# 4-5. Larger selected spatial 2x2 panels; SpatialPCA colorbar label has extra room.
plotted_spatial = plot_selected_spatial_panels(results, adata_full)

# 6. Baseline 2x2 composite: SpatialPCA, Popari, STAMP, MEFISTO.
plot_baseline_spatial_composite(results, adata_full, plotted_spatial)

# 7. Horizontal T cell near/far violin with a shared y-axis label.
plot_near_far_violin_tcell_horizontal()

# 8. Larger 34.5 mo benchmark clustering panel for mouse 101.
plot_benchmark_clustering_mouse101(results, adata_full)

# 9. Mouse 97 Slingshot cluster/pseudotime spatial panel, aligned to the benchmark panel.
plot_mouse97_slingshot_spatial_for_figure_drawing()

# 10. Microglia stGP2 enrichment panel from the saved enrichment CSV.
plot_microglia_stgp2_enrichment_for_figure_drawing()

# 11. Proximity OLS regression coefficient panel for Microglia stGP2.
plot_microglia_stgp2_proximity_regression()

# 12. 3D spacetime stack of the Microglia stGP2 spatial embedding.
plot_microglia_stgp2_spacetime_stack(results, adata_full)

print(f'All revised mouse brain figures saved to: {OUT_DIR}')